# Gikomba Stall Analysis: Wanjiku's Fresh-Produce Business

**Analyst:** Nathan Karondo | **Cohort:** <YOUR COHORT>
**Tools:** Core Python only (lists, dictionaries, tuples, functions, f-strings). No pandas or NumPy.

**Business question:** How is Wanjiku's business doing? Is it growing? What should she restock, and where should she focus?

The first code cell below holds the data exactly as provided. Every later cell reads from it, so nothing is typed in twice.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics and best/worst days

**Approach:** Use `sum()`, `len()`, `max()` and `min()` for the four statistics. Then use `list.index()` to find the position of the highest and lowest values. Positions 0-6 are week 1, so the week number is `position // 7 + 1`, and the day name is `days[position]`.

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_pos = daily_sales.index(highest_sales)
worst_pos = daily_sales.index(lowest_sales)
best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales   : KES {total_sales:,}")
print(f"Average daily : KES {average_sales:,.2f}")
print(f"Highest day   : KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day    : KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales   : KES 46,450
Average daily : KES 3,317.86
Highest day   : KES 5,600 (Sat, week 2)
Lowest day    : KES 1,950 (Tue, week 1)


## Task 2: Week totals and week-on-week growth

**Approach:** `week_totals()` slices the list into `[:7]` (week 1) and `[7:14]` (week 2) and returns both sums as a tuple. The caller unpacks the tuple, then growth % = (week 2 - week 1) / week 1 x 100.

In [3]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for a list of 14 daily sales.

    Week 1 is the first 7 values (slice [:7]) and week 2 is the next 7
    (slice [7:14]).

    Args:
        sales: list of 14 daily sales amounts (KES).

    Returns:
        tuple: (week1_total, week2_total)
    """
    return sum(sales[:7]), sum(sales[7:14])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total : KES {week1_total:,}")
print(f"Week 2 total : KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:+.2f}%")

Week 1 total : KES 22,050
Week 2 total : KES 24,400
Week-on-week growth: +10.66%


## Task 3: Classifying every day as Strong, Normal or Weak

**Approach:** `classify_day()` compares a day's sales with the average. More than 20% above average (above `average x 1.2`) is "Strong", more than 20% below (below `average x 0.8`) is "Weak", anything else is "Normal". I apply it to every day and count the categories with the dictionary counting pattern (`counts[label] = counts.get(label, 0) + 1`).

In [4]:
def classify_day(amount, average):
    """Classify one day's sales relative to the average.

    Args:
        amount: sales for the day (KES).
        average: average daily sales (KES).

    Returns:
        str: "Strong" if amount is more than 20% above average,
             "Weak" if more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.20:
        return "Strong"
    if amount < average * 0.80:
        return "Weak"
    return "Normal"


day_labels = [classify_day(amount, average_sales) for amount in daily_sales]

category_counts = {}
for label in day_labels:
    category_counts[label] = category_counts.get(label, 0) + 1

for i, (day, amount, label) in enumerate(zip(days, daily_sales, day_labels), start=1):
    print(f"Day {i:>2} ({day}): KES {amount:>5,}  ->  {label}")

print()
for label in ("Strong", "Normal", "Weak"):
    print(f"{label:<7}: {category_counts.get(label, 0)} days")

Day  1 (Mon): KES 2,300  ->  Weak
Day  2 (Tue): KES 1,950  ->  Weak
Day  3 (Wed): KES 2,800  ->  Normal
Day  4 (Thu): KES 2,100  ->  Weak
Day  5 (Fri): KES 3,600  ->  Normal
Day  6 (Sat): KES 5,200  ->  Strong
Day  7 (Sun): KES 4,100  ->  Strong
Day  8 (Mon): KES 2,500  ->  Weak
Day  9 (Tue): KES 2,200  ->  Weak
Day 10 (Wed): KES 3,100  ->  Normal
Day 11 (Thu): KES 2,400  ->  Weak
Day 12 (Fri): KES 3,900  ->  Normal
Day 13 (Sat): KES 5,600  ->  Strong
Day 14 (Sun): KES 4,700  ->  Strong

Strong : 4 days
Normal : 4 days
Weak   : 6 days


## Task 4: Weekday versus weekend average

**Approach:** Loop over the two lists together with `zip()`. Sat and Sun go in the weekend list, everything else in the weekday list. Then compare the two averages: difference % = (weekend avg - weekday avg) / weekday avg x 100.

In [5]:
weekday_sales, weekend_sales = [], []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}  ({len(weekday_sales)} days)")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}  ({len(weekend_sales)} days)")
print(f"Weekend sells {weekend_premium_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00  (10 days)
Weekend average (Sat-Sun): KES 4,900.00  (4 days)
Weekend sells 82.5% more per day than weekdays


## Task 5: Reorder list and gross margin per product

**Approach:** `products_to_reorder()` loops through `stock.items()` and collects every product whose `qty` is at or below its `reorder` level (`<=`). A second loop calculates gross margin % = (price - cost) / price x 100 for each product and stores it in a dictionary.

In [6]:
def products_to_reorder(stock_dict):
    """Return the products whose stock is at or below the reorder level.

    Args:
        stock_dict: dict of {product: {"qty", "reorder", "cost", "price"}}.

    Returns:
        list: names of products where qty <= reorder.
    """
    return [name for name, info in stock_dict.items()
            if info["qty"] <= info["reorder"]]


reorder_list = products_to_reorder(stock)

margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print("Products to reorder:", ", ".join(reorder_list))
print()
print(f"{'Product':<10}{'Stock (kg)':>11}{'Reorder (kg)':>14}{'Margin %':>10}")
for name, info in stock.items():
    print(f"{name:<10}{info['qty']:>11}{info['reorder']:>14}{margins[name]:>9.1f}%")

Products to reorder: Tomatoes, Sukuma

Product    Stock (kg)  Reorder (kg)  Margin %
Tomatoes           18            25     33.3%
Onions             40            20     30.0%
Sukuma              9            15     40.0%
Potatoes           55            30     33.3%
Avocado            12            10     42.9%


## Task 6: Stall Performance Report

**Approach:** Print everything from the variables calculated in Tasks 1-5, using f-strings. The `:,` format gives thousands separators, `:<` and `:>` align the columns, and `:.1f` fixes the decimals.

In [7]:
line = "=" * 52
thin = "-" * 52

print(line)
print(f"{'STALL PERFORMANCE REPORT':^52}")
print(f"{'Wanjiku - Gikomba Market (14 days)':^52}")
print(line)

print("SALES SUMMARY")
print(f"  {'Total sales':<26}KES {total_sales:>12,}")
print(f"  {'Average daily sales':<26}KES {average_sales:>12,.2f}")
print(f"  {'Best day':<26}KES {highest_sales:>12,}  ({best_day}, wk {best_week})")
print(f"  {'Worst day':<26}KES {lowest_sales:>12,}  ({worst_day}, wk {worst_week})")
print(thin)

print("GROWTH")
print(f"  {'Week 1 total':<26}KES {week1_total:>12,}")
print(f"  {'Week 2 total':<26}KES {week2_total:>12,}")
print(f"  {'Week-on-week growth':<26}    {growth_pct:>+11.1f}%")
print(thin)

print("DAY CLASSIFICATION")
for label in ("Strong", "Normal", "Weak"):
    print(f"  {label:<26}{category_counts.get(label, 0):>2} days")
print(thin)

print("WEEKDAY vs WEEKEND (average per day)")
print(f"  {'Weekdays (Mon-Fri)':<26}KES {weekday_avg:>12,.2f}")
print(f"  {'Weekends (Sat-Sun)':<26}KES {weekend_avg:>12,.2f}")
print(f"  {'Weekend advantage':<26}    {weekend_premium_pct:>+11.1f}%")
print(thin)

print("STOCK AND MARGINS")
print(f"  {'Product':<10}{'Stock':>7}{'Reorder':>9}{'Margin':>9}  Status")
for name, info in stock.items():
    status = "REORDER" if name in reorder_list else "OK"
    print(f"  {name:<10}{info['qty']:>5}kg{info['reorder']:>7}kg{margins[name]:>8.1f}%  {status}")
print(thin)
print(f"  Restock now: {', '.join(reorder_list)}")
print(line)

              STALL PERFORMANCE REPORT              
         Wanjiku - Gikomba Market (14 days)         
SALES SUMMARY
  Total sales               KES       46,450
  Average daily sales       KES     3,317.86
  Best day                  KES        5,600  (Sat, wk 2)
  Worst day                 KES        1,950  (Tue, wk 1)
----------------------------------------------------
GROWTH
  Week 1 total              KES       22,050
  Week 2 total              KES       24,400
  Week-on-week growth                 +10.7%
----------------------------------------------------
DAY CLASSIFICATION
  Strong                     4 days
  Normal                     4 days
  Weak                       6 days
----------------------------------------------------
WEEKDAY vs WEEKEND (average per day)
  Weekdays (Mon-Fri)        KES     2,685.00
  Weekends (Sat-Sun)        KES     4,900.00
  Weekend advantage                   +82.5%
----------------------------------------------------
STOCK AND MARGINS
  P

## Recommendations for Wanjiku

1. **Keep going: the business is growing.** Sales rose from KES 22,050 in week 1 to KES 24,400 in week 2, which is +10.7% in a single week.
2. **Put your effort into weekends.** Saturday and Sunday average KES 4,900 a day against KES 2,685 on weekdays, which is 82.5% more, and all 4 "Strong" days were weekends. Arrive with full stock on Friday evening so no weekend sale is lost to empty crates.
3. **Restock Tomatoes and Sukuma now.** Tomatoes (18 kg against a 25 kg reorder level) and Sukuma (9 kg against 15 kg) are both below reorder level, and running out before the weekend would lose her best days.
4. **Push Avocado and Sukuma for profit.** Avocado has the highest margin at 42.9% (KES 60 profit per kg) and Sukuma follows at 40.0%. Onions have the lowest margin at 30.0%, so she should not tie up cash in extra Onions (40 kg, already double their reorder level of 20 kg).
5. **Fix the slow weekdays.** 6 of the 14 days were "Weak" (below KES 2,654), and the lowest day was a Tuesday at KES 1,950. A small weekday offer, such as a bundle of Tomatoes, Onions and Sukuma, could lift these days towards the KES 2,685 weekday average.